In [0]:
from pyspark.sql import functions as F, DataFrame
from datetime import datetime

# ── Configuração ──────────────────────────────────────────────────────────────
SOURCE_TABLE = "homologacao.salutar_saude.raw_utilizacao"
TARGET_TABLE = "homologacao.salutar_saude.silver_utilizacao"

DATE_COLS    = ["data_evento"]                        # datas mistas: YYYY-MM-DD, DD/MM/YYYY, DD-MM-YYYY
INITCAP_COLS = ["tipo_evento", "especialidade"]       # initcap + trim proativo
TRIM_COLS    = []                                     # sem campos de texto livre nesta tabela
PRICE_COLS   = ["valor_sinistro"]                     # decimal(12,2) — formatos mistos na fonte
MERGE_KEY    = "evento_id"                            # chave primária para o MERGE incremental

# Valores conhecidos de tipo_evento (pós-initcap) — [WARN] sem assert, lista pode crescer
TIPOS_CONHECIDOS = [
    "Consulta", "Exame", "Pronto-socorro", "Terapia",  # initcap não capitaliza após hífen: 'pronto-socorro' → 'Pronto-socorro'
    "Internação", "Cirurgia", "Exame De Imagem",
]

run_ts = datetime.now()

print(f"Pipeline  : silver_utilizacao")
print(f"Iniciado  : {run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Origem    : {SOURCE_TABLE}")
print(f"Destino   : {TARGET_TABLE}")

Pipeline  : silver_utilizacao
Iniciado  : 2026-07-04 19:17:20
Origem    : homologacao.salutar_saude.raw_utilizacao
Destino   : homologacao.salutar_saude.silver_utilizacao


In [0]:
df_raw = spark.table(SOURCE_TABLE)
n_raw = df_raw.count()

print(f"[OK] {n_raw:,} registros lidos de {SOURCE_TABLE}")

[OK] 6,465 registros lidos de homologacao.salutar_saude.raw_utilizacao


In [0]:
def parse_date(col_name: str) -> F.Column:
    """Normaliza datas para YYYY-MM-DD.
    Suporta os formatos: YYYY-MM-DD, DD-MM-YYYY, DD/MM/YYYY.
    Usa try_to_date para retornar NULL em vez de lançar exceção.
    """
    return F.date_format(
        F.coalesce(
            F.expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd-MM-yyyy')"),
            F.expr(f"try_to_date(`{col_name}`, 'dd/MM/yyyy')"),
        ),
        "yyyy-MM-dd",
    ).alias(col_name)


def parse_initcap(col_name: str) -> F.Column:
    """Normaliza campos categóricos: remove espaços extras e padroniza capitalização.
    Não usar em siglas/códigos. Ex.: 'CONSULTA' → 'Consulta' | 'pronto-socorro' → 'Pronto-Socorro'.
    Nota: 'Exame de Imagem' → 'Exame De Imagem' (initcap capitaliza cada palavra).
    """
    return F.initcap(F.trim(F.col(col_name))).alias(col_name)


def parse_trim(col_name: str) -> F.Column:
    """Remove espaços extras de campos de texto livres (nome, descrição)."""
    return F.trim(F.col(col_name)).alias(col_name)


def parse_preco(col_name: str) -> F.Column:
    """Normaliza valores monetários para decimal(12,2).
    Reutiliza a lógica de parse_valor_mensal/parse_preco — mesmos 3 formatos na fonte:
      - 'R$ 126.080,78'  → prefixo + ponto milhar + vírgula decimal
      - '108,03'         → sem prefixo, vírgula decimal
      - '95500.69'       → sem prefixo, ponto decimal
    Se contém vírgula: remove 'R$ ', remove pontos (milhar), troca vírgula por ponto.
    Senão: remove 'R$ ' e usa ponto já como decimal.
    """
    c      = F.trim(F.regexp_replace(F.col(col_name), r"R\$\s*", ""))
    br_fmt = F.regexp_replace(F.regexp_replace(c, r"\.", ""), ",", ".")
    return (
        F.when(F.col(col_name).contains(","), br_fmt)
         .otherwise(c)
         .cast("decimal(12,2)")
         .alias(col_name)
    )


def transform(df: DataFrame, cols: list) -> DataFrame:
    """Aplica todas as transformações de padronização da camada silver.
    Remove a coluna _rescued_data (artefato da camada RAW) e
    adiciona _updated_at com o timestamp de execução.

    Args:
        df  : DataFrame de origem (camada RAW).
        cols: lista de colunas pré-computada fora da função (evita RPC repetido
              ao acessar df.columns dentro de um loop/função no Spark Connect).
    """
    return df.select(
        *[
            parse_date(c)    if c in DATE_COLS    else
            parse_initcap(c) if c in INITCAP_COLS else
            parse_preco(c)   if c in PRICE_COLS   else
            parse_trim(c)    if c in TRIM_COLS    else
            F.col(c)
            for c in cols
        ],
        F.lit(run_ts).cast("timestamp").alias("_updated_at"),  # em select, não withColumn
    )


print("[OK] Funções de transformação definidas.")

[OK] Funções de transformação definidas.


In [0]:
# Computa schema uma única vez fora da função (evita RPC repetido no Spark Connect)
silver_cols = [c for c in df_raw.columns if c != "_rescued_data"]
df_silver = transform(df_raw, silver_cols)

# ── Validação de qualidade ────────────────────────────────────────────────────
DQ_COLS = DATE_COLS + INITCAP_COLS + TRIM_COLS + PRICE_COLS

null_counts = (
    df_silver
    .select(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in DQ_COLS])
    .collect()[0]
    .asDict()
) if DQ_COLS else {}

n_silver = df_silver.count()

# [WARN] sem assert: tipo_evento é lista aberta no negócio (novos tipos podem surgir)
unexpected_tipo = df_silver.filter(
    F.col("tipo_evento").isNotNull()
    & ~F.col("tipo_evento").isin(*TIPOS_CONHECIDOS)
).count()
new_tipos = (
    df_silver
    .filter(F.col("tipo_evento").isNotNull() & ~F.col("tipo_evento").isin(*TIPOS_CONHECIDOS))
    .select("tipo_evento").distinct()
    .collect()
) if unexpected_tipo > 0 else []
new_tipos = [r["tipo_evento"] for r in new_tipos]

# Duplicatas de chave na fonte (serão removidas no passo de escrita)
dup_keys_df  = df_silver.groupBy(MERGE_KEY).count().filter(F.col("count") > 1)
n_dup_keys   = dup_keys_df.count()
dup_ids_list = [str(r[MERGE_KEY]) for r in dup_keys_df.collect()] if n_dup_keys > 0 else []

print("── Data Quality ───────────────────────────────────────────────────────────")
print(f"  Total de registros           : {n_silver:,}")
print(f"  Correspondência com RAW      : {'[OK]' if n_silver == n_raw else '[WARN] divergência!'}")
for col_name, nulls in null_counts.items():
    tag = "[WARN]" if nulls > 0 else "[OK]  "
    print(f"  {tag} {col_name}: {nulls:,} nulos")
tag = "[WARN]" if unexpected_tipo > 0 else "[OK]  "
new_detail = f" → novos tipos: {new_tipos}" if new_tipos else ""
print(f"  {tag} tipo_evento desconhecidos      : {unexpected_tipo:,}{new_detail}")
tag = "[WARN]" if n_dup_keys > 0 else "[OK]  "
dup_detail = f" → ids: {dup_ids_list}" if dup_ids_list else ""
print(f"  {tag} {MERGE_KEY} duplicados na fonte : {n_dup_keys:,}{dup_detail}")
print("─" * 65)

# Asserts bloqueantes: apenas problemas críticos
assert n_silver == n_raw, f"Contagem divergente: RAW={n_raw}, silver={n_silver}"
# Nota: unexpected_tipo é [WARN] sem assert — lista de tipos pode crescer no negócio.
# Se um novo tipo aparecer, adicione-o em TIPOS_CONHECIDOS na célula de Configuração.

── Data Quality ───────────────────────────────────────────────────────────
  Total de registros           : 6,465
  Correspondência com RAW      : [OK]
  [OK]   data_evento: 0 nulos
  [OK]   tipo_evento: 0 nulos
  [OK]   especialidade: 0 nulos
  [OK]   valor_sinistro: 0 nulos
  [OK]   tipo_evento desconhecidos      : 0
  [OK]   evento_id duplicados na fonte : 0
─────────────────────────────────────────────────────────────────


In [0]:
from delta.tables import DeltaTable

# ── Escrita incremental via MERGE ─────────────────────────────────────────────
# Estratégia de deduplicacão — 2 etapas:
# 1. dropDuplicates()            → remove linhas 100% idênticas
# 2. dropDuplicates([MERGE_KEY]) → garante cardinalidade 1:1 por chave para o MERGE Delta
df_to_merge = df_silver.dropDuplicates().dropDuplicates([MERGE_KEY])

if spark.catalog.tableExists(TARGET_TABLE):
    delta_target = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_target.alias("target")
        .merge(
            df_to_merge.alias("source"),
            f"target.{MERGE_KEY} = source.{MERGE_KEY}"
        )
        .whenMatchedUpdateAll()       # atualiza evento existente se houver mudança
        .whenNotMatchedInsertAll()    # insere novo evento
        # .whenNotMatchedBySourceDelete()  # descomente para remover eventos deletados da RAW
        .execute()
    )
    print(f"[OK] MERGE concluído      : {TARGET_TABLE}")
else:
    # Primeira execução: cria a tabela (carga inicial)
    df_to_merge.write.format("delta").saveAsTable(TARGET_TABLE)
    print(f"[OK] Carga inicial        : {TARGET_TABLE}")

n_written = spark.table(TARGET_TABLE).count()
print(f"[OK] Registros na tabela  : {n_written:,}")
print(f"[OK] Fim do pipeline      : {datetime.now():%Y-%m-%d %H:%M:%S}")

[OK] Carga inicial        : homologacao.salutar_saude.silver_utilizacao
[OK] Registros na tabela  : 6,465
[OK] Fim do pipeline      : 2026-07-04 19:16:34


In [0]:
%sql
SELECT * FROM homologacao.salutar_saude.silver_utilizacao
ORDER BY evento_id
LIMIT 50

evento_id,beneficiario_id,data_evento,tipo_evento,especialidade,valor_sinistro,_updated_at
1,1101,2025-06-11,Pronto-socorro,Oncologia,691.39,2026-07-04T19:15:12.243Z
2,3248,2024-10-15,Consulta,Dermatologia,372.24,2026-07-04T19:15:12.243Z
3,1332,2024-08-26,Consulta,Psiquiatria,276.86,2026-07-04T19:15:12.243Z
4,1333,2025-04-06,Cirurgia,Cardiologia,95500.69,2026-07-04T19:15:12.243Z
5,187,2024-10-24,Exame,Oncologia,108.03,2026-07-04T19:15:12.243Z
6,956,2025-08-10,Exame,Ortopedia,581.80,2026-07-04T19:15:12.243Z
7,1033,2025-06-27,Pronto-socorro,Endocrinologia,2220.65,2026-07-04T19:15:12.243Z
8,1548,2025-02-24,Cirurgia,Ortopedia,126080.78,2026-07-04T19:15:12.243Z
9,134,2025-01-05,Exame,Endocrinologia,918.97,2026-07-04T19:15:12.243Z
10,407,2025-02-20,Exame,Dermatologia,285.08,2026-07-04T19:15:12.243Z
